# Eigenmode + EPR Analysis Notebook
### Transmon + Readout Resonator — HFSS Eigenmode Simulation and Energy-Participation-Ratio Quantization

**Scope.** This notebook parallels the Q3D capacitance-extraction / LOM notebook, but replaces the
electrostatic (Q3D) simulation + Lumped-Oscillator-Model (LOM) extraction with a full-wave
**HFSS eigenmode simulation** + **Energy Participation Ratio (EPR)** extraction, using
`qiskit-metal`'s `EPRanalysis`

**Design scope for this notebook.** Single transmon qubit + single readout resonator
(no multi-qubit coupling — that is deferred to a follow-on notebook once $g$, $\chi_{ij}$
between *two* qubits is in scope).

**Structure**
1. Theory — eigenmode simulation, EPR quantization
2. Environment & imports
3. Design setup (`QDesign`: transmon + readout resonator)
4. Renderer & simulation setup (`EPRanalysis` → HFSS)
5. Run the eigenmode simulation
6. Raw eigenmode results & sanity checks
7. EPR setup & junction participation
8. Hamiltonian diagonalization & parameter extraction
9. EPR-specific sanity checks
10. `EPRFullResult` dataclass
11. Demo / smoke-test harness
12. Appendix A — LOM vs. EPR cross-check


## 1. Theory

### 1.1 Why a different simulation from Q3D/LOM

The Q3D pipeline solves an **electrostatic (quasi-static) problem**: fixed conductors at
prescribed potentials, Laplace's equation $\nabla^2\phi = 0$ in the dielectric, and the output is
a **Maxwell capacitance matrix** $C_{ij}$ — a purely geometric, frequency-independent object.
The LOM (Lumped Oscillator Model) pipeline then *builds* the qubit and resonator as lumped
$LC$ (and Josephson) elements on top of that matrix, and diagonalizes a lumped-element
Hamiltonian.

That approach is excellent when the circuit is well described by lumped elements coupled
through the capacitance matrix, but it cannot by itself capture full-wave effects: radiative
loss into the resonator's own mode structure, standing-wave resonator physics, package/box
mode participation, or any physics that depends on the *spatial distribution of the fields*
rather than just terminal capacitances. For that, we go to a **full-wave eigenmode**
simulation of the entire electromagnetic structure, and extract circuit parameters from the
**field solution** rather than from a capacitance matrix.

### 1.2 The eigenmode problem in HFSS

HFSS's eigenmode solver looks for **source-free** solutions of the vector wave equation. Starting
from the frequency-domain Maxwell curl equations in a source-free, linear, lossless (to first
order) medium,

$$
\nabla\times\mathbf E = i\omega\mu_0\mathbf H, \qquad
\nabla\times\mathbf H = -i\omega\varepsilon\,\mathbf E,
$$

eliminating $\mathbf H$ gives the vector Helmholtz equation

$$
\nabla\times\left(\frac{1}{\mu_0}\nabla\times\mathbf E\right) - \omega^2\varepsilon\,\mathbf E = 0 .
$$

Finite-element discretization (vector/edge elements on the mesh) turns this into a
**generalized matrix eigenvalue problem**

$$
K\,\mathbf e = \omega^2 M\,\mathbf e,
$$

where $K$ is the curl-curl (stiffness) matrix built from $1/\mu_0$, $M$ is the mass matrix built
from $\varepsilon(\mathbf r)$, and $\mathbf e$ is the vector of edge-basis-function coefficients
representing $\mathbf E(\mathbf r)$. Solving this eigenproblem for the lowest $N$ eigenvalues
gives the resonant (angular) frequencies $\omega_m$ and their associated 3D field patterns
$\mathbf E_m(\mathbf r)$, $\mathbf H_m(\mathbf r)$ — these are the **normal modes of the whole
electromagnetic structure**, not of any individual lumped element.

This is conceptually very different from Q3D: there is no excitation, no port, and no
capacitance matrix. The "circuit" here is whatever standing-wave patterns the geometry
supports, and coupling between what you *think of* as separate elements (qubit pad, resonator,
package) is encoded implicitly in how much each mode's field overlaps each piece of geometry.

**Boundary conditions** matter as much as they did for Q3D, but the failure modes look
different:
- **Perfect-E** boundaries (ground planes, chip metal) enforce $\mathbf E_\parallel = 0$.
- **Radiation / PML** boundaries approximate open space; if placed too close to the chip they
  perturb mode frequencies and (if you run a loss-aware sweep) artificially load down the $Q$.
- An **unmerged or missing conductor** (e.g. a floating ground plane, an unassigned package
  wall) can silently introduce a spurious low-frequency box mode, or shift the frequency of the
  physical mode you care about — this is the eigenmode-solver analog of the Q3D "unlisted
  conductor" row-sum diagnostic, and we build an equivalent sanity check for it below (§6).

**Convergence** is judged by adaptive mesh refinement: HFSS refines the mesh near regions of
highest field-energy density between passes and reports $\Delta f$ (or $\Delta S$) between
successive passes for each requested mode. We treat convergence with the same discipline as Q3D
mesh convergence — track the per-pass frequency history, don't trust a single-pass result.

### 1.3 Representing the Josephson junction in the eigenmode solve

The Josephson junction's true potential is nonlinear, $U(\varphi) = -E_J\cos\varphi$. The
eigenmode solver, however, only handles **linear** media and boundary conditions. The
standard approach (and the one `qiskit-metal`/`pyEPR` automate) is:

1. In the eigenmode solve, replace each junction by a **linear lumped inductor**
   $L_J = \left(\dfrac{\hbar}{2e}\right)^{\!2}\dfrac{1}{E_J}$ (the junction's *linearized*,
   small-oscillation inductance around $\varphi=0$), realized in HFSS as a lumped RLC boundary
   condition across the junction gap. Junction self-capacitance $C_J$, if included, is a second
   lumped element in parallel.
2. Solve the resulting **linear** eigenmode problem to get mode frequencies $\omega_m$ and field
   patterns.
3. Recover the leading-order nonlinear correction to those modes **afterward**, perturbatively,
   using the Energy Participation Ratio method (§1.4) — this is where the $\cos\varphi$
   nonlinearity re-enters the calculation.

This is the key conceptual difference from LOM: in LOM, you build a nonlinear circuit
Hamiltonian directly out of extracted lumped elements ($C$, $E_J$) and diagonalize it (e.g. with
`scqubits`). In EPR, you first solve the *linear* full-wave problem, and then treat the
junction nonlinearity as a **perturbation on the linear normal modes** of that solution.

### 1.4 Energy Participation Ratio (EPR) theory

For each eigenmode $m$ and each junction (or, more generally, each weakly nonlinear element)
$j$, the **energy participation ratio** is defined as the fraction of that mode's total
inductive energy that is stored in junction $j$'s linear inductance, at the classical
field-solution level:

$$
p_{mj} \;=\; \frac{U_{L_j,m}}{U_{\mathrm{ind,\,total},\,m}}
\;=\;\frac{\tfrac12 L_j\,I_{j,m}^2}{U_{\mathrm{ind,\,total},\,m}} ,
\qquad 0 \le p_{mj} \le 1 \ \ (\text{with sign convention for multi-junction interference}).
$$

$I_{j,m}$ is the peak current through the $j$-th junction's lumped inductor when the structure
rings at mode $m$ with the field solution normalized to a single photon's worth of energy;
$U_{\mathrm{ind,\,total},\,m}$ is the total inductive (equivalently, by energy balance in a
lossless resonant mode, equal to the total electric) energy of that mode. In practice `pyEPR`
computes $p_{mj}$ directly from the field solution's stored magnetic/electric energy densities
integrated over the junction's lumped-element region, normalized against the total mode energy
— this is exactly analogous in spirit to your Q3D row-sum check: **it is a self-consistency
diagnostic as much as it is a physical quantity** (participations across all lossy/nonlinear
elements plus the "rest of the mode" should sum to something physically sensible, and any
single $p_{mj}$ outside $[0,1]$, or an implausibly small $p_{mj}$ for a mode you *expect* to be
qubit-like, signals a meshing, geometry, or boundary problem before you trust the extracted
Hamiltonian).

Physically, $p_{mj}$ tells you **how much of mode $m$'s zero-point energy lives across junction
$j$** — i.e., how strongly that particular linear mode "feels" the nonlinear element. A mode
that is almost entirely capacitive-pad-to-ground with negligible current through the junction
has $p_{mj}\approx 0$ and will show almost no Kerr/anharmonicity correction; a mode dominated by
oscillating current through the junction (the "qubit-like" mode) has $p_{mj}$ close to 1.

### 1.5 From participation ratios to the quantum Hamiltonian

Once the participation matrix $P = [p_{mj}]$ (modes $\times$ junctions) is known, `pyEPR`
promotes the classical linear eigenmodes to quantum harmonic oscillators (each mode $m$ becomes
a mode with bosonic operators $a_m, a_m^\dagger$ and bare frequency $\omega_m$ read directly off
the eigenmode solve), and re-inserts the **junction nonlinearity** as a perturbation expressed
in terms of those mode operators. Schematically, the phase across junction $j$ is expanded as a
sum over the zero-point fluctuations of every mode weighted by its participation,

$$
\hat\varphi_j \;=\; \sum_m \varphi_{\mathrm{zpf},mj}\,(a_m + a_m^\dagger), \qquad
\varphi_{\mathrm{zpf},mj} \;\propto\; \sqrt{p_{mj}\,\frac{\hbar\omega_m}{2E_{J,j}}},
$$

and the junction's $-E_J\cos\hat\varphi_j$ potential is Taylor-expanded in $\hat\varphi_j$. The
quadratic term simply renormalizes the bare mode frequencies (a small frequency shift from the
"already linear" inductance you fed into HFSS); the **quartic term** is what produces the
physics you actually want out of this notebook:

- **Self-Kerr / anharmonicity** of the qubit-like mode, $\alpha \equiv \chi_{qq}$ — the same
  anharmonicity you also get from LOM/`scqubits`, but now derived from the full-wave field
  solution instead of a lumped-element capacitance matrix.
- **Cross-Kerr / dispersive shift** $\chi_{mn}$ between any two modes sharing participation in
  the same junction (e.g. qubit–resonator $\chi$) — this is exactly the quantity your deferred
  two-qubit-coupling topic ($g$, $\chi$) will generalize to *two qubits* sharing participation
  through a coupler.
- The exact numerical prefactors and higher-order (6th-order and beyond) corrections follow the
  systematic energy-participation quantization procedure of Minev *et al.*, "Energy-Participation
  Quantization of Josephson Circuits" (npj Quantum Information, 2021) — `pyEPR`'s
  `QuantumAnalysis` implements this diagonalization for you (`analyze_hamiltonian`); we treat it
  as the ground-truth reference for the exact coefficients rather than re-deriving them by hand
  here, and we validate our extracted numbers against it directly in §8.

**Bottom line for this notebook's data flow:**

$$
\underbrace{\text{HFSS eigenmode solve}}_{\omega_m,\ \text{field patterns}}
\;\longrightarrow\;
\underbrace{\text{EPR extraction}}_{P = [p_{mj}]}
\;\longrightarrow\;
\underbrace{\text{Hamiltonian diagonalization}}_{f_{01},\ \alpha,\ \chi_{mn}}
$$

which is the full-wave analog of your existing

$$
\underbrace{\text{Q3D electrostatic solve}}_{C_{ij}}
\;\longrightarrow\;
\underbrace{\text{LOM extraction}}_{E_J/E_C,\ \text{lumped Hamiltonian}}
\;\longrightarrow\;
\underbrace{\text{scqubits diagonalization}}_{f_{01},\ \alpha}
$$

pipeline. Section 12 (Appendix A) cross-checks the two final answers against each other.

### 1.6 What EPR gives you that LOM does not (and vice versa)

| | LOM (Q3D-based) | EPR (HFSS-based) |
|---|---|---|
| Field solve | Electrostatic (quasi-static) | Full-wave eigenmode |
| Primary extracted object | Capacitance matrix $C_{ij}$ | Participation matrix $p_{mj}$ |
| Captures resonator standing-wave physics | No (resonator modeled as lumped $LC$ you supply) | Yes (resonator is itself a solved eigenmode) |
| Captures radiative / package modes | No | Yes, if included in geometry/box |
| Nonlinearity handling | Built into lumped Hamiltonian directly (exact diagonalization via `scqubits`) | Perturbative expansion in $\hat\varphi_j$ around linear modes |
| Compute cost | Cheap (electrostatic solve) | Expensive (full-wave, multiple adaptive passes per mode) |
| Best for | Fast design-space sweeps, early-stage sizing | Final verification, cross-Kerr/coupling extraction, anything sensitive to spatial field structure |

In practice you'll want both in the optimization-library pipeline: LOM for fast inner-loop
parameter sweeps, EPR as the higher-fidelity verification step before committing to a mask.


## 2. Environment & Imports

Requires a local Ansys Electronics Desktop installation reachable via COM automation
(Windows, or a configured remote-desktop bridge), `qiskit-metal`, and `pyEPR` (installed
automatically as a `qiskit-metal` dependency). `qiskit-metal`'s `EPRanalysis` class drives the
HFSS renderer and `pyEPR` for you — there is no separate manual "connect to project" step
required for the standard flow.

In [1]:
import numpy as np
import pandas as pd
from dataclasses import dataclass, field
from typing import Optional
import matplotlib.pyplot as plt

import qiskit_metal as metal
from qiskit_metal import designs, draw
from qiskit_metal.qlibrary.qubits.transmon_pocket import TransmonPocket
from qiskit_metal.qlibrary.tlines.meandered import RouteMeander
from qiskit_metal.qlibrary.terminations.open_to_ground import OpenToGround
from qiskit_metal.analyses.quantization import EPRanalysis

import pyEPR as epr

print("qiskit-metal:", metal.__version__)
print("pyEPR:", epr.__version__)


qiskit-metal: 0.7.5
pyEPR: 1.0.1


## 3. Design Setup

Single transmon + single quarter-wave (or half-wave) readout resonator, coupled capacitively.
Replace the geometric parameters below with your actual design values — this mirrors the same
`QDesign` you'd use to drive the Q3D capacitance extraction, so if you already have a design
object from that notebook you can load/reuse it directly instead of rebuilding it here.

In [2]:
design = designs.DesignPlanar()
design.overwrite_enabled = True

# --- Transmon qubit -------------------------------------------------------
transmon_options = dict(
    pos_x='0um', pos_y='0um',
    pad_width='425um', pad_height='150um', pad_gap='30um',
    connection_pads=dict(
        readout=dict(loc_W=1, loc_H=1, pad_width='60um', pad_gap='10um')
    ),
    hfss_inductance='10nH',   # linearized L_J used for the eigenmode solve (see 1.3)
    hfss_capacitance='2fF',   # junction self-capacitance C_J
)
q1 = TransmonPocket(design, 'Q1', options=transmon_options)

# --- Readout resonator (meandered, open-to-ground termination) -----------
otg = OpenToGround(design, 'open_to_ground', options=dict(pos_x='2000um', pos_y='0um', orientation='0'))

readout_options = dict(
    pin_inputs=dict(
        start_pin=dict(component='Q1', pin='readout'),
        end_pin=dict(component='open_to_ground', pin='open'),
    ),
    lead=dict(start_straight='100um'),
    fillet='90um',
    total_length='6mm',      # tune for target resonator frequency
)
readout = RouteMeander(design, 'readout', options=readout_options)

gui = metal.MetalGUI(design)
gui.rebuild()
gui.autoscale()


## 4. Renderer & Simulation Setup

`EPRanalysis(design, "hfss")` instantiates the HFSS renderer *and* wires up the `pyEPR`
`DistributedAnalysis`/`QuantumAnalysis` machinery underneath a single object, so simulation
setup, running, and EPR post-processing all happen through the same `eig_qres` handle.

In [3]:
eig_qres = EPRanalysis(design, "hfss")

# --- Eigenmode setup -------------------------------------------------------
hfss = eig_qres.sim.renderer
hfss.start()

setup = eig_qres.sim.setup
setup.name          = "Setup"
setup.min_freq_ghz   = 1.0     # search window lower bound
setup.n_modes        = 2       # qubit-like + resonator-like mode
setup.max_delta_f    = 0.5     # % frequency delta between adaptive passes -> convergence
setup.max_passes     = 20
setup.min_passes     = 3
setup.min_converged  = 2
setup.basis_order    = -1      # mixed order, HFSS default recommendation for eigenmode

# Design variables actually swept in HFSS (so Lj can be tuned/optimized later without
# re-rendering geometry -- important for the eventual optimization-library sweep loop).
setup.vars = dict(Lj='10 nH', Cj='2 fF')

print(setup)


INFO 10:44AM [connect_project]: Connecting to Ansys Desktop API...
INFO 10:44AM [load_ansys_project]: 	Opened Ansys App
INFO 10:44AM [load_ansys_project]: 	Opened Ansys Desktop v2025.1.4
INFO 10:44AM [load_ansys_project]: 	Opened Ansys Project
	Folder:    C:/Users/DELL/Documents/Ansoft/
	Project:   Test_design_project
INFO 10:44AM [connect_design]: 	Opened active design
	Design:    q_0_eigenmode_hfss [Solution type: Eigenmode]
INFO 10:44AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 10:44AM [connect]: 	Connected to project "Test_design_project" and design "q_0_eigenmode_hfss" 😀 

INFO 10:44AM [connect_design]: 	Opened active design
	Design:    transmon_readout_eigenmode [Solution type: Eigenmode]
WARNING 10:44AM [connect_setup]: 	No design setup detected.
WARNING 10:44AM [connect_setup]: 	Creating eigenmode default setup.
INFO 10:44AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)


{'name': 'Setup', 'reuse_selected_design': True, 'reuse_setup': True, 'min_freq_ghz': 1.0, 'n_modes': 2, 'max_delta_f': 0.5, 'max_passes': 20, 'min_passes': 3, 'min_converged': 2, 'pct_refinement': 30, 'basis_order': -1, 'vars': {'Lj': '10 nH', 'Cj': '2 fF'}}


**Boundary / box sanity note (§1.2):** before running, confirm the render box, chip
size, and radiation boundary offsets are consistent with your Q3D notebook's converged box
size — an inconsistency here is the eigenmode-solver analog of the Q3D "unlisted conductor"
failure mode and will bias every downstream EPR number.

## 5. Running the Eigenmode Simulation

`eig_qres.sim.run(...)` renders the specified components into the currently active HFSS design,
assigns the lumped RLC junction boundary automatically from each `TransmonPocket`'s
`hfss_inductance`/`hfss_capacitance` options, and launches the adaptive eigenmode solve.

In [4]:
eig_qres.sim.run(
    name='TransmonReadoutEig',
    components=['Q1', 'readout', 'open_to_ground'],
    open_terminations=[('open_to_ground', 'open')],
    box_plus_buffer=True,
)

# Convergence history per adaptive pass -- inspect before trusting the result (cf. Q3D mesh
# convergence discipline).
convergence_df = eig_qres.sim.renderer.get_convergences()
convergence_df


INFO 10:46AM [connect_design]: 	Opened active design
	Design:    TransmonReadoutEig_hfss [Solution type: Eigenmode]
WARNING 10:46AM [connect_setup]: 	No design setup detected.
WARNING 10:46AM [connect_setup]: 	Creating eigenmode default setup.
INFO 10:46AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 10:46AM [get_setup]: 	Opened setup `Setup`  (<class 'pyEPR.ansys.HfssEMSetup'>)
INFO 10:46AM [analyze]: Analyzing setup Setup
10:50AM 47s INFO [get_f_convergence]: Saved convergences to c:\Users\DELL\chip-design\metal-flow\hfss_eig_f_convergence.csv
10:50AM 48s INFO [get_f_convergence]: Saved convergences to c:\Users\DELL\chip-design\metal-flow\hfss_eig_f_convergence.csv


(             Solved Elements  Max Delta Freq. %
 Pass Number                                    
 1                       9819                NaN
 2                      12768           26.08300
 3                      14681           23.07300
 4                      16893            7.53890
 5                      20980            9.53610
 6                      25637            2.07610
 7                      33014            2.03890
 8                      42688            1.04090
 9                      55352            0.44311
 10                     71864            0.51219
 11                     93337            0.31220
 12                    121251            0.52311
 13                    157503            0.32629
 14                    204635            0.22137,
          re(Mode(1)) [g]  re(Mode(2)) [g]
 Pass []                                  
 1               3.091945         5.700933
 2               3.898413         6.681275
 3               4.797911         7.630322


In [7]:
convergence_df[0]

,Solved Elements,Max Delta Freq. %
Pass Number,,
1,9819,NaN
2,12768,26.08300
3,14681,23.07300
4,16893,7.53890
5,20980,9.53610
6,25637,2.07610
7,33014,2.03890
8,42688,1.04090
9,55352,0.44311


## 6. Raw Eigenmode Results & Sanity Checks

Extract the raw (linear, pre-EPR) mode frequencies and quality factors, and run the
eigenmode-solver analog of the Q3D row-sum diagnostic.

In [6]:
convergence_df[1]

,re(Mode(1)) [g],re(Mode(2)) [g]
Pass [],,
1,3.091945,5.700933
2,3.898413,6.681275
3,4.797911,7.630322
4,5.077887,8.205566
5,5.325125,8.988060
6,5.435679,9.137220
7,5.503818,9.323515
8,5.540259,9.420565
9,5.564809,9.448843


In [13]:
eig_qres.sim.close()

## 7. EPR Setup & Junction Participation

Point `pyEPR` at the junction geometry (the rectangle representing the linear inductor and the
line defining its current direction) and the design variables it corresponds to, then compute
the participation matrix.

In [10]:
jj_setup = eig_qres.setup.junctions
jj_setup['jj'] = dict(
    rect        = 'JJ_rect_Lj_Q1_rect_jj',   # HFSS object name of the lumped-RLC rectangle
    line        = 'JJ_Lj_Q1_rect_jj_',       # HFSS polyline defining current direction
    Lj_variable = 'Lj',
    Cj_variable = 'Cj',
)

eig_qres.run_epr()




INFO 10:56AM [__init__]: Design "TransmonReadoutEig_hfss" info:
INFO 10:56AM [__init__]: 	# eigenmodes    2
	# variations    1


INFO 10:56AM [__init__]: Design "TransmonReadoutEig_hfss" info:
INFO 10:56AM [__init__]: 	# eigenmodes    2
	# variations    1
INFO 10:56AM [do_EPR_analysis]: Variation 0  [1/1]



        energy_elec_all       = 2.08111209390502e-24
        energy_elec_substrate = 1.91384775551798e-24
        EPR of substrate = 92.0%

        energy_mag    = 2.0802898816216e-24
        energy_mag % of energy_elec_all  = 100.0%
        


INFO 10:56AM [do_EPR_analysis]: Mode 0 at 5.67 GHz   [1/2]
INFO 10:56AM [do_EPR_analysis]:     Calculating E_magnetic ...
INFO 10:56AM [do_EPR_analysis]:     Calculating E_electric ...
INFO 10:56AM [do_EPR_analysis]:      (E_E-E_H)/E_E=  99.4%   E_E=1.388e-24   E_H=7.882e-27
INFO 10:56AM [do_EPR_analysis]:     Calculating junction EPR, method=`line_voltage`
INFO 10:56AM [do_EPR_analysis]: 	junction        EPR p_0j   sign s_0j    (p_capacitive)
INFO 10:56AM [calc_p_junction]: 	jj               1.01887  (+)        0.0258723
INFO 10:56AM [calc_p_junction]: 		(U_tot_cap-U_tot_ind)/mean=0.06%
INFO 10:56AM [get_Qdielectric]: Calculating Qdielectric_main for mode 0 (0/1)
INFO 10:56AM [get_Qdielectric]: p_dielectric_main_0 = 0.9182509806891188
INFO 10:56AM [do_EPR_analysis]: Mode 1 at 9.48 GHz   [2/2]
INFO 10:56AM [do_EPR_analysis]:     Calculating E_magnetic ...
INFO 10:56AM [do_EPR_analysis]:     Calculating E_electric ...
INFO 10:56AM [do_EPR_analysis]:      (E_E-E_H)/E_E=   0.0%   E_E=1.04

      0.99
   0.00043
       243     0.35
      0.35 0.000126
    268+0j 0.322+0j
  0.322+0j0.000104+0j


...

###### Numerical diagonalization

Lj,10
0,5416.09+ 0.00j
1,9476.43+ 0.00j


#### Kerr Non-linear coefficient table (MHz)

###### Numerical diagonalization

0                               1
Lj                                                  
10 0  267.80+  0.00j  3.22e-01+0.00e+            00j
   1    0.32+  0.00j  1.04e-04+0.00e+            00j

## 8. Hamiltonian Diagonalization & Parameter Extraction

`pyEPR`'s `QuantumAnalysis` takes the participation matrix plus the linear mode frequencies and
performs the perturbative (or, optionally, numerically exact within the truncated Hilbert
space) diagonalization described in §1.5, returning corrected frequencies, anharmonicities, and
the cross-Kerr/$\chi$ matrix.

In [11]:
quantum_analysis = eig_qres.data.quantum_analysis if hasattr(eig_qres.data, 'quantum_analysis') \
    else epr.QuantumAnalysis(eig_qres.data.get_epr_dataframes())  # fallback explicit construction

quantum_analysis.analyze_all_variations(cos_trunc=8, fock_trunc=7)

f_ND, chi_ND = quantum_analysis.results.get_frequencies_HFSS(), quantum_analysis.results.get_chi_matrix()

f01_corrected_ghz = f_ND.iloc[0, 0]        # qubit-like mode, first variation
alpha_mhz         = chi_ND.iloc[0, 0] * 1e3  # diagonal chi_qq = anharmonicity (GHz -> MHz)
chi_qr_mhz        = chi_ND.iloc[0, 1] * 1e3  # off-diagonal qubit-resonator dispersive shift

print(f"Corrected f01           : {f01_corrected_ghz:.4f} GHz")
print(f"Anharmonicity alpha      : {alpha_mhz:.2f} MHz")
print(f"Qubit-resonator chi      : {chi_qr_mhz:.3f} MHz")


AttributeError: 'EPRanalysis' object has no attribute 'data'

## 9. Sanity Checks Section

Consolidated diagnostics, mirroring the structure of the Q3D notebook's diagnostics section.

In [ ]:
def full_epr_sanity_report(convergence_df, raw_modes, participation_matrix,
                            expected_f01_ghz=None, expected_fres_ghz=None):
    report = {}
    report['convergence'] = sanity_check_convergence(convergence_df)
    report['mode_warnings'] = sanity_check_mode_sanity(raw_modes, expected_f01_ghz, expected_fres_ghz)
    report['participation_issues'] = sanity_check_participation(participation_matrix)
    return report

report = full_epr_sanity_report(convergence_df, raw_modes, participation_matrix,
                                 expected_f01_ghz=5.0, expected_fres_ghz=7.0)
report


## 10. `EPRFullResult` Dataclass

Structurally consistent with `LOMFullResult` from `lom_full_analysis.py`, so downstream code
(e.g. the LOM-vs-EPR cross-check in the Appendix, or the optimization library) can treat the two
pipelines' outputs uniformly.

In [ ]:
@dataclass
class EPRFullResult:
    # Raw eigenmode outputs
    mode_frequencies_ghz: pd.Series
    mode_q_factors: pd.Series
    convergence_history: pd.DataFrame

    # EPR outputs
    participation_matrix: pd.DataFrame

    # Extracted Hamiltonian parameters
    f01_ghz: float
    anharmonicity_mhz: float
    chi_matrix_mhz: pd.DataFrame

    # Metadata
    junction_variables: dict = field(default_factory=dict)
    sanity_report: dict = field(default_factory=dict)
    notes: Optional[str] = None


def build_epr_full_result(raw_modes, convergence_df, participation_matrix,
                           quantum_analysis, setup_vars, sanity_report,
                           notes=None) -> EPRFullResult:
    f_ND = quantum_analysis.results.get_frequencies_HFSS()
    chi_ND = quantum_analysis.results.get_chi_matrix()
    return EPRFullResult(
        mode_frequencies_ghz = raw_modes['f_GHz'] if 'f_GHz' in raw_modes else raw_modes.iloc[:, 0],
        mode_q_factors       = raw_modes['Q'] if 'Q' in raw_modes else raw_modes.iloc[:, 1],
        convergence_history  = convergence_df,
        participation_matrix = participation_matrix,
        f01_ghz              = float(f_ND.iloc[0, 0]),
        anharmonicity_mhz    = float(chi_ND.iloc[0, 0] * 1e3),
        chi_matrix_mhz       = chi_ND * 1e3,
        junction_variables   = setup_vars,
        sanity_report        = sanity_report,
        notes                = notes,
    )

epr_result = build_epr_full_result(
    raw_modes, convergence_df, participation_matrix, quantum_analysis,
    setup_vars=dict(Lj='10 nH', Cj='2 fF'), sanity_report=report,
    notes="Single transmon + single readout resonator, first pass at design point."
)
epr_result


## 11. Demo / Test Harness

Smoke tests analogous to the `lom_full_analysis.py` test harness: checks on output shapes,
signs, and physically-expected ordering, runnable end-to-end against either a live Ansys
session or a cached/pickled `EPRFullResult` for fast CI-style testing.

In [ ]:
def smoke_test_epr_full_result(result: EPRFullResult):
    assert result.f01_ghz > 0, "f01 must be positive"
    assert result.anharmonicity_mhz < 0, "Transmon anharmonicity should be negative"
    assert (result.participation_matrix.values >= -1e-6).all(), "Participation ratios should not be meaningfully negative for a single-junction device"
    assert (result.participation_matrix.values <= 1 + 1e-6).all(), "Participation ratios should not exceed 1"
    assert result.chi_matrix_mhz.shape[0] == result.chi_matrix_mhz.shape[1], "Chi matrix must be square"
    assert not result.sanity_report['mode_warnings'], f"Unresolved mode warnings: {result.sanity_report['mode_warnings']}"
    print("All smoke tests passed.")

smoke_test_epr_full_result(epr_result)


## Appendix A: LOM vs. EPR Cross-Check

Cross-checks the EPR-derived $f_{01}$ / anharmonicity against the LOM pipeline's output for the
*same physical design point*, as a consistency test between the two independent extraction
routes described in §1.6. Load (or recompute) the `LOMFullResult` for this design's capacitance
matrix and junction parameters, then compare.

In [ ]:
# from lom_full_analysis import LOMFullResult, run_lom_full_analysis  # your existing module

# Placeholder -- replace with an actual LOMFullResult for the matching design point.
# lom_result = run_lom_full_analysis(...)

def compare_lom_epr(lom_result, epr_result, f01_tol_mhz=50, alpha_tol_mhz=10):
    df = pd.DataFrame({
        'LOM':  [lom_result.f01_ghz * 1e3, lom_result.anharmonicity_mhz],
        'EPR':  [epr_result.f01_ghz * 1e3, epr_result.anharmonicity_mhz],
    }, index=['f01_MHz', 'anharmonicity_MHz'])
    df['delta_MHz'] = df['EPR'] - df['LOM']

    flags = []
    if abs(df.loc['f01_MHz', 'delta_MHz']) > f01_tol_mhz:
        flags.append(f"f01 mismatch exceeds {f01_tol_mhz} MHz tolerance -- check that both "
                      f"pipelines use the same Lj/Cj and geometry, or that the EPR box/mesh "
                      f"has converged.")
    if abs(df.loc['anharmonicity_MHz', 'delta_MHz']) > alpha_tol_mhz:
        flags.append(f"Anharmonicity mismatch exceeds {alpha_tol_mhz} MHz tolerance -- often "
                      f"indicates residual meshing error in the EPR participation ratio, or "
                      f"that the LOM capacitance matrix's box size differs from the EPR box.")
    return df, flags

# lom_epr_comparison, lom_epr_flags = compare_lom_epr(lom_result, epr_result)
# lom_epr_comparison
